# Round-Two Post-Advisor OI Oral-Dental Reanalysis

**Date:** 2026-04-18  
**Run ID:** 20260418_1037_post_advisor_round2  
**SEED:** 20260228  
**Data Source Authority:** canonical (post_advisor_round2_v1_2026-04-18)  
**N:** 34 subjects  

---

## Purpose

This notebook performs the complete round-two post-advisor analysis of the OI oral-dental cohort using the revised post-advisor semantic layer. It is paired with a standalone Python script (`oi_oro_dental_post_advisor_round2_reanalysis_v1.py`) for deterministic reproducibility.

Key semantic revisions from round-one:
- Primary Angle variable: `angle_sinifi_clean` (values 1/2/3 or missing; infraocclusion case excluded)
- Caries endpoint: `caries_count_total` (alias for `dmft_dmft` count-like measure)
- Anomaly endpoint: `doku_anomalisi_any` (binary presence)
- New secondary endpoint: `di_any` (binary DI presence)

**Non-primary endpoints:** Angle-class distributions are descriptive only due to sparse cell counts.

In [ ]:
!pip install -q numpy pandas scipy statsmodels


In [ ]:
!mkdir -p /content/oi_round2_bundle
!unzip -o /content/oi_round2_post_advisor_colab_bundle_20260418.zip -d /content/oi_round2_bundle


Archive:  /content/oi_round2_post_advisor_colab_bundle_20260418.zip
  inflating: /content/oi_round2_bundle/oi_round2_post_advisor_colab_bundle_20260418/bundle_root/OI_POST_ADVISOR_DATA_SEMANTICS_AND_ROUND2_REANALYSIS_STATUS_REPORT.md  
  inflating: /content/oi_round2_bundle/oi_round2_post_advisor_colab_bundle_20260418/bundle_root/COLAB_README.md  
  inflating: /content/oi_round2_bundle/oi_round2_post_advisor_colab_bundle_20260418/bundle_root/colab_bootstrap_round2.py  
  inflating: /content/oi_round2_bundle/oi_round2_post_advisor_colab_bundle_20260418/bundle_root/bundle_file_manifest.txt  
  inflating: /content/oi_round2_bundle/oi_round2_post_advisor_colab_bundle_20260418/bundle_root/data_decisions_post_advisor_round2.md  
  inflating: /content/oi_round2_bundle/oi_round2_post_advisor_colab_bundle_20260418/bundle_root/.claude/CLAUDE.md  
  inflating: /content/oi_round2_bundle/oi_round2_post_advisor_colab_bundle_20260418/bundle_root/.github/copilot-instructions.md  
  inflating: /content

In [ ]:
%cd /content/oi_round2_bundle/oi_round2_post_advisor_colab_bundle_20260418/bundle_root


/content/oi_round2_bundle/oi_round2_post_advisor_colab_bundle_20260418/bundle_root


In [ ]:
!find 02_analysis/scripts/validation -maxdepth 1 -type f | sort


02_analysis/scripts/validation/oi_oro_dental_post_advisor_round2_reanalysis_v1_fast.py
02_analysis/scripts/validation/oi_oro_dental_post_advisor_round2_reanalysis_v1.py


In [ ]:
!python 02_analysis/scripts/validation/oi_oro_dental_post_advisor_round2_reanalysis_v1.py


✓ Data loaded: N=34
✓ Post-advisor semantic fields validated
✓ Infraocclusion case (id=5) correctly marked

=== DESCRIPTIVE ANALYSIS ===
{
  "n_total": 34,
  "age_mean": 10.470588235294118,
  "age_sd": 5.332776263241952,
  "age_range": [
    2,
    18
  ],
  "gene_group_distribution": {
    "P3H1": 8,
    "FKBP10": 8,
    "COL1A2": 7,
    "COL1A1": 6,
    "Other": 5
  },
  "dentition_stage_distribution": {
    "1": 8,
    "2": 14,
    "3": 12
  },
  "endpoints": {
    "doku_anomalisi_any": {
      "n_cases": 10,
      "prevalence": 0.29411764705882354
    },
    "gingivitis": {
      "n_cases": 11,
      "prevalence": 0.3235294117647059
    },
    "caries_any": {
      "n_cases": 24,
      "prevalence": 0.7058823529411765
    },
    "infraokluzyon_var_clean": {
      "n_cases": 1,
      "prevalence": 0.029411764705882353
    }
  },
  "caries_count_total": {
    "mean": 3.088235294117647,
    "median": 1.5,
    "sd": 3.8953557977986875,
    "range": [
      0,
      14
    ]
  }
}

=== 

## Stage 1: Setup & Determinism

In [ ]:
import numpy as np
import pandas as pd
import random
import os
import json
import warnings
from pathlib import Path
from datetime import datetime
from scipy import stats as sp_stats
from scipy.stats import chi2_contingency, mannwhitneyu

# Determinism (non-negotiable)
SEED = 20260228
np.random.seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

warnings.filterwarnings('ignore', category=FutureWarning)

print(f"✓ SEED set to {SEED}")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ Pandas version: {pd.__version__}")


## Stage 2: Input Paths & Semantic Authority

In [ ]:
# Workspace paths (absolute, notebook-independent)
REPO_ROOT = Path.cwd()  # Assume notebook runs from repo root or adjust
if not REPO_ROOT.name == 'modest-euler-d7d141':  # Worktree detection
    # Fallback: derive from notebook location
    REPO_ROOT = Path('/Users/centaurioun/Repos/osteogenesis_imperfecta/.claude/worktrees/modest-euler-d7d141')

DATA_INPUT = REPO_ROOT / "01_data/derived/osteogenesis_imperfecta_analysis_ready_post_advisor_round2_v1_2026-04-18.csv"
OUTPUT_DIR = REPO_ROOT / "03_outputs/reports/run_20260418_1037_post_advisor_round2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Data input: {DATA_INPUT}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"✓ Data input exists: {DATA_INPUT.exists()}")
print(f"✓ Output directory exists: {OUTPUT_DIR.exists()}")


## Stage 3: Data Loading & Semantic Validation (QC)

In [ ]:
# Load post-advisor dataset
df = pd.read_csv(DATA_INPUT)
df_clean = df.copy()

print(f"✓ Data loaded: N={len(df)}")
print(f"\nColumns in dataset: {list(df.columns)}")

# Critical validations
assert len(df) == 34, f"Expected N=34, got {len(df)}"
assert df['semantic_version'].unique()[0] == 'post_advisor_round2_v1_2026-04-18', "Wrong semantic version"
assert df['source_authority'].unique()[0] == 'canonical', "Wrong source authority"

# Post-advisor semantic field validation
print("\n=== POST-ADVISOR SEMANTIC VALIDATION ===")
print(f"angle_sinifi_clean non-missing values: {df['angle_sinifi_clean'].notna().sum()}")
assert df['angle_sinifi_clean'].dropna().astype(int).isin([1, 2, 3]).all(), "angle_sinifi_clean invalid"
print("✓ angle_sinifi_clean contains only 1/2/3 (or missing)")

infra_case = df[df['occl_tip'] == 4]
print(f"\nInfraocclusion case (occl_tip=4):")
if len(infra_case) > 0:
    print(f"  - hasta_kodu: {infra_case['hasta_kodu'].values[0]}")
    print(f"  - angle_sinifi_clean: {infra_case['angle_sinifi_clean'].values[0]}")
    print(f"  - infraokluzyon_var_clean: {infra_case['infraokluzyon_var_clean'].values[0]}")
    assert pd.isna(infra_case['angle_sinifi_clean'].values[0]), "Infraocclusion must have NA angle_sinifi_clean"
    assert infra_case['infraokluzyon_var_clean'].values[0] == 1, "Infraocclusion must have infraokluzyon_var_clean=1"
    print("✓ Infraocclusion case correctly marked (angle excluded, infraokluzyon=1)")
else:
    print("  - No infraocclusion case found")

assert (df['caries_count_total'] == df['dmft_dmft']).all(), "caries_count_total != dmft_dmft"
print("✓ caries_count_total = dmft_dmft (count-like caries measure)")

assert (df['doku_anomalisi_any'] == (df['doku_anomalisi'] != 0).astype(int)).all(), "doku_anomalisi_any derivation failed"
print("✓ doku_anomalisi_any = (doku_anomalisi != 0) (binary anomaly presence)")

assert (df['di_any'] == (df['doku_anomalisi'] == 2).astype(int)).all(), "di_any derivation failed"
print("✓ di_any = (doku_anomalisi == 2) (binary DI presence)")

print("\n✓ ALL SEMANTIC VALIDATIONS PASSED")


## Stage 4: Descriptive Analysis

In [ ]:
print("=== COHORT DESCRIPTIVES ===")
print(f"N: {len(df_clean)}")
print(f"Age (median, IQR): {df_clean['yas'].median()} ({df_clean['yas'].quantile(0.25)}-{df_clean['yas'].quantile(0.75)})")
print(f"Age (range): {df_clean['yas'].min()}-{df_clean['yas'].max()} years")

print("\n=== DENTITION STAGE ===")
dentition_counts = df_clean['dentition_donemi_clean'].value_counts().sort_index()
for stage, count in dentition_counts.items():
    stage_labels = {1: 'Deciduous/Mixed (<6)', 2: 'Mixed (6-<14)', 3: 'Permanent (≥14)'}
    print(f"  Stage {stage} ({stage_labels[stage]}): {count}")

print("\n=== GENE GROUP DISTRIBUTION ===")
gene_counts = df_clean['gen_group'].value_counts().sort_values(ascending=False)
for gene, count in gene_counts.items():
    pct = 100 * count / len(df_clean)
    note = " (SAP threshold n≥6)" if count < 6 else ""
    print(f"  {gene}: {count} ({pct:.1f}%){note}")

print("\n=== PRIMARY ENDPOINT PREVALENCE ===")
endpoints = {
    'doku_anomalisi_any': 'Any dental anomaly',
    'gingivitis': 'Gingivitis',
    'caries_any': 'Any caries',
    'infraokluzyon_var_clean': 'Infraocclusion',
    'di_any': 'DI presence (secondary)',
}

for var, label in endpoints.items():
    n_cases = int(df_clean[var].sum())
    pct = 100 * df_clean[var].mean()
    print(f"  {label}: {n_cases}/{len(df_clean)} ({pct:.1f}%)")

print("\n=== CONTINUOUS ENDPOINTS ===")
print(f"  Caries count_total (median, IQR): {df_clean['caries_count_total'].median()} ({df_clean['caries_count_total'].quantile(0.25)}-{df_clean['caries_count_total'].quantile(0.75)})")
print(f"  Caries count_total (mean ± SD): {df_clean['caries_count_total'].mean():.2f} ± {df_clean['caries_count_total'].std():.2f}")


## Stage 5: Feasibility Gate

Before inferential analysis, confirm each endpoint is feasible under observed data structure.

In [ ]:
print("=== FEASIBILITY ASSESSMENT ===")

# Primary binary endpoints
for endpoint in ['doku_anomalisi_any', 'gingivitis', 'caries_any']:
    ct = pd.crosstab(df_clean[endpoint], df_clean['gen_group'])
    n_cases = int(df_clean[endpoint].sum())
    min_cell = ct.min().min()
    status = "✓ Feasible" if n_cases >= 2 else "⚠ Sparse"
    print(f"\n{endpoint}:")
    print(f"  Cases: {n_cases}/{len(df_clean)}")
    print(f"  Min cell in gene group table: {min_cell}")
    print(f"  Status: {status}")
    if min_cell < 5:
        print(f"  → Use permutation χ² (Fisher-Freeman-Halton exact if available)")

# Infraocclusion special case
print(f"\ninfraokluzyon_var_clean:")
print(f"  Cases: {int(df_clean['infraokluzyon_var_clean'].sum())}/{len(df_clean)}")
print(f"  Status: ⚠ Descriptive/sensitivity only (n=1)")
print(f"  → Not a primary inferential endpoint; preserve case in descriptive tables")

# Continuous
print(f"\ncaries_count_total:")
print(f"  Test: Kruskal-Wallis (mandatory per SAP for continuous)")
print(f"  Pairwise: Mann-Whitney U + Holm correction")
print(f"  Status: ✓ Feasible")

# Angle class (descriptive only)
angle_eligible = df_clean['angle_sinifi_clean'].notna().sum()
angle_ct = pd.crosstab(df_clean['angle_sinifi_clean'], df_clean['gen_group'], margins=True)
print(f"\nangle_sinifi_clean:")
print(f"  Eligible cases (non-missing): {angle_eligible}")
print(f"  Class distribution: Class I {(df_clean['angle_sinifi_clean']==1).sum()}, Class II {(df_clean['angle_sinifi_clean']==2).sum()}, Class III {(df_clean['angle_sinifi_clean']==3).sum()}")
print(f"  Status: ⚠ Descriptive only (sparse cell structure due to Class I dominance: 81.8%)")
print(f"  → Report descriptively; do not route through primary inferential family")

print("\n=== FEASIBILITY SUMMARY ===")
print("✓ Primary binary endpoints: Feasible (doku_anomalisi_any, gingivitis, caries_any)")
print("✓ Primary continuous endpoint: Feasible (caries_count_total)")
print("⚠ Secondary binary endpoint: Sparse cells (di_any, n=7)")
print("⚠ Infraocclusion: Descriptive only (n=1)")
print("⚠ Angle class: Descriptive only (sparse cell structure)")


## Stage 6: Primary Inferential Analysis

In [ ]:
# Helper function: Exact χ² + Permutation χ²
def exact_chi2_and_permutation(outcome, groupvar, n_permutations=10000, seed=SEED):
    """Compute chi-square and permutation chi-square p-values."""
    np.random.seed(seed)
    ct = pd.crosstab(outcome, groupvar)

    # Standard chi-square
    try:
        chi2, pval_chi2, dof, expected = chi2_contingency(ct)
    except:
        chi2, pval_chi2, dof = np.nan, np.nan, np.nan
        expected = None

    # Cramer's V
    n = len(outcome)
    cramer_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1))) if chi2 >= 0 else np.nan

    # Permutation test
    perm_chi2_dist = []
    for _ in range(n_permutations):
        perm_outcome = np.random.permutation(outcome.values)
        perm_ct = pd.crosstab(perm_outcome, groupvar)
        try:
            perm_chi2, _, _, _ = chi2_contingency(perm_ct)
            perm_chi2_dist.append(perm_chi2)
        except:
            pass

    pval_perm = (np.array(perm_chi2_dist) >= chi2).mean() if perm_chi2_dist else np.nan

    return {
        'chi2': chi2,
        'pval_chi2': pval_chi2,
        'cramer_v': cramer_v,
        'pval_permutation': pval_perm,
        'has_small_cells': (expected < 5).any() if expected is not None else False,
    }

print("=== PRIMARY INFERENTIAL RESULTS BY GENE GROUP ===")
print(f"\nPermutation iterations: {10000}")
print(f"SEED: {SEED}")
print(f"Multiple comparison correction: Holm")

# Collect primary family results
primary_results = {}
for endpoint in ['doku_anomalisi_any', 'gingivitis', 'caries_any']:
    result = exact_chi2_and_permutation(df_clean[endpoint], df_clean['gen_group'])
    primary_results[endpoint] = result

    print(f"\n{endpoint}:")
    print(f"  χ² = {result['chi2']:.4f}")
    print(f"  p (classic) = {result['pval_chi2']:.4f}")
    print(f"  p (permutation) = {result['pval_permutation']:.4f}")
    print(f"  Cramer's V = {result['cramer_v']:.4f}")
    print(f"  Small cells detected: {result['has_small_cells']}")

print(f"\nNote: Holm correction will be applied to the family: {{doku_anomalisi_any, gingivitis, caries_any}}")


## Stage 7: Kruskal-Wallis for Caries Count

In [ ]:
# Kruskal-Wallis test (mandatory for continuous by SAP)
groups = [df_clean[df_clean['gen_group'] == g]['caries_count_total'].values
          for g in sorted(df_clean['gen_group'].unique())]

kw_h, kw_p = sp_stats.kruskal(*groups)

# Epsilon-squared effect size
n = len(df_clean)
k = len(groups)
epsilon_sq = (kw_h - k + 1) / (n - k)
epsilon_sq = max(0, min(1, epsilon_sq))

print("=== CARIES COUNT (Continuous) ===")
print(f"\nKruskal-Wallis test across gene groups:")
print(f"  H = {kw_h:.4f}")
print(f"  p = {kw_p:.4f}")
print(f"  Epsilon-squared (effect size) = {epsilon_sq:.4f}")
print(f"  Groups (k) = {k}")
print(f"  N = {n}")

# Pairwise Mann-Whitney U
print(f"\nPairwise Mann-Whitney U tests (with Holm correction):")
from itertools import combinations

gene_groups = sorted(df_clean['gen_group'].unique())
pairwise_results = []
for g1, g2 in combinations(gene_groups, 2):
    data1 = df_clean[df_clean['gen_group'] == g1]['caries_count_total']
    data2 = df_clean[df_clean['gen_group'] == g2]['caries_count_total']
    u_stat, p_val = mannwhitneyu(data1, data2, alternative='two-sided')
    pairwise_results.append({
        'group1': g1,
        'group2': g2,
        'u_stat': u_stat,
        'p': p_val,
    })
    print(f"  {g1} vs {g2}: U={u_stat:.1f}, p={p_val:.4f}")

# Holm correction
p_vals = [r['p'] for r in pairwise_results]
from scipy.stats import rankdata
sorted_idx = np.argsort(p_vals)
m = len(p_vals)
holm_corrected = np.zeros(m)
for i, idx in enumerate(sorted_idx):
    holm_corrected[idx] = min(1.0, p_vals[idx] * (m - i))

print(f"\n  Holm-corrected p-values:")
for r, p_h in zip(pairwise_results, holm_corrected):
    print(f"    {r['group1']} vs {r['group2']}: p_holm = {p_h:.4f}")


## Stage 8: Robustness - Leave-One-Out Stability

In [ ]:
print("=== LEAVE-ONE-OUT (LOO) STABILITY ===")
print("\nRerunning primary endpoints with each subject excluded:")
print("Reporting: baseline p, LOO p_min, LOO p_max, delta_p_max")

loo_summary = {}
for endpoint in ['doku_anomalisi_any', 'gingivitis', 'caries_any']:
    baseline = exact_chi2_and_permutation(df_clean[endpoint], df_clean['gen_group'])
    baseline_p = baseline['pval_chi2']

    loo_pvals = []
    for idx in range(len(df_clean)):
        outcome_loo = df_clean[endpoint].drop(idx)
        groupvar_loo = df_clean['gen_group'].drop(idx)
        try:
            result_loo = exact_chi2_and_permutation(outcome_loo, groupvar_loo)
            loo_pvals.append(result_loo['pval_chi2'])
        except:
            pass

    if loo_pvals:
        p_min = np.min(loo_pvals)
        p_max = np.max(loo_pvals)
        delta_p_max = abs(p_max - baseline_p)
        loo_summary[endpoint] = {
            'baseline_p': baseline_p,
            'loo_p_min': p_min,
            'loo_p_max': p_max,
            'delta_p_max': delta_p_max,
        }
        print(f"\n{endpoint}:")
        print(f"  Baseline p: {baseline_p:.4f}")
        print(f"  LOO p_min: {p_min:.4f}")
        print(f"  LOO p_max: {p_max:.4f}")
        print(f"  Max Δp: {delta_p_max:.4f}")

print("\n✓ LOO analysis complete")


## Stage 9: Generate Publication Tables

In [ ]:
# Table 1: Overall descriptives
table1_data = [
    ['N', 34, '—'],
    ['Age (median, IQR)', f"{df_clean['yas'].median()} ({df_clean['yas'].quantile(0.25)}-{df_clean['yas'].quantile(0.75)})", '—'],
    ['Angle Class I', f"{(df_clean['angle_sinifi_clean']==1).sum()} ({100*(df_clean['angle_sinifi_clean']==1).sum()/angle_eligible:.1f}%)", 'Among N=33 eligible'],
    ['Angle Class II', f"{(df_clean['angle_sinifi_clean']==2).sum()} ({100*(df_clean['angle_sinifi_clean']==2).sum()/angle_eligible:.1f}%)", 'Among N=33 eligible'],
    ['Angle Class III', f"{(df_clean['angle_sinifi_clean']==3).sum()} ({100*(df_clean['angle_sinifi_clean']==3).sum()/angle_eligible:.1f}%)", 'Among N=33 eligible'],
    ['Infraocclusion', f"{int(df_clean['infraokluzyon_var_clean'].sum())} ({100*df_clean['infraokluzyon_var_clean'].mean():.1f}%)", '—'],
    ['Dental Anomaly (any)', f"{int(df_clean['doku_anomalisi_any'].sum())} ({100*df_clean['doku_anomalisi_any'].mean():.1f}%)", '—'],
    ['Gingivitis', f"{int(df_clean['gingivitis'].sum())} ({100*df_clean['gingivitis'].mean():.1f}%)", '—'],
    ['Caries (any)', f"{int(df_clean['caries_any'].sum())} ({100*df_clean['caries_any'].mean():.1f}%)", '—'],
    ['Caries count (median, IQR)', f"{df_clean['caries_count_total'].median()} ({df_clean['caries_count_total'].quantile(0.25)}-{df_clean['caries_count_total'].quantile(0.75)})", '—'],
]

table1_df = pd.DataFrame(table1_data, columns=['Variable', 'Value', 'Note'])
table1_path = OUTPUT_DIR / 'publication_table1_overall_round2.csv'
table1_df.to_csv(table1_path, index=False)
print(f"✓ Table 1 (Overall): {table1_path}")
print(table1_df.to_string(index=False))


## Stage 10: Export Results & Manifests

In [ ]:
# Run manifest
import hashlib
import platform
import statsmodels

run_manifest = {
    'run_id': '20260418_1037_post_advisor_round2',
    'timestamp': datetime.utcnow().isoformat(),
    'seed': SEED,
    'permutation_iters': 10000,
    'dataset_path': str(DATA_INPUT),
    'dataset_name': DATA_INPUT.name,
    'dataset_hash': hashlib.sha256(DATA_INPUT.read_bytes()).hexdigest(),
    'semantic_version': 'post_advisor_round2_v1_2026-04-18',
    'source_authority': 'canonical',
    'n_subjects': len(df_clean),
    'python_version': platform.python_version(),
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'scipy_version': scipy_version,
    'statsmodels_version': statsmodels.__version__,
}

manifest_path = OUTPUT_DIR / 'run_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)

print(f"✓ Run manifest: {manifest_path}")
print(json.dumps(run_manifest, indent=2, default=str))


## Final: File Registry & Issue Log

In [ ]:
print("\n" + "="*70)
print("ROUND-TWO POST-ADVISOR REANALYSIS — EXECUTION SUMMARY")
print("="*70)
print(f"Run ID: 20260418_1037_post_advisor_round2")
print(f"Date: {datetime.utcnow().isoformat()}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Data: {DATA_INPUT.name}")
print(f"N subjects: {len(df_clean)}")
print(f"SEED: {SEED}")
print(f"Semantic version: post_advisor_round2_v1_2026-04-18")
print("="*70)

print("\nGeneratedArtifacts:")
for file in OUTPUT_DIR.glob('*'):
    if file.is_file():
        size = file.stat().st_size
        print(f"  - {file.name} ({size} bytes)")

print("\n✓ NOTEBOOK EXECUTION COMPLETE")


In [ ]:
import os
import sys
from pathlib import Path

runtime_info = {
    "python_executable": sys.executable,
    "cwd": str(Path.cwd()),
    "platform": sys.platform,
    "has_google_colab_module": False,
    "is_colab_hint": False,
}

try:
    import google.colab  # type: ignore
    runtime_info["has_google_colab_module"] = True
except Exception:
    runtime_info["has_google_colab_module"] = False

runtime_info["is_colab_hint"] = (
    runtime_info["has_google_colab_module"]
    or "COLAB_" in " ".join(os.environ.keys())
    or str(Path('/content').exists())
)

runtime_info


{'python_executable': '/usr/bin/python3',
 'cwd': '/content',
 'platform': 'linux',
 'has_google_colab_module': True,
 'is_colab_hint': True}

In [4]:
import zipfile
from pathlib import Path

zip_name = "oi_round2_post_advisor_colab_bundle_20260418.zip"
content_dir = Path("/content")
zip_path = content_dir / zip_name

if not zip_path.exists():
    try:
        from google.colab import files  # type: ignore
        print(f"Upload required: {zip_name}")
        uploaded = files.upload()
        if zip_name not in uploaded:
            # Allow fallback to first uploaded zip
            zips = [k for k in uploaded.keys() if k.endswith('.zip')]
            if not zips:
                raise RuntimeError("No zip uploaded.")
            zip_path = content_dir / zips[0]
        else:
            zip_path = content_dir / zip_name
    except Exception as exc:
        raise RuntimeError(
            f"Zip not found at {zip_path} and interactive upload is unavailable. "
            "Please upload the prepared bundle zip to /content and re-run this cell."
        ) from exc

extract_root = content_dir
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_root)

bundle_root = extract_root / "oi_round2_post_advisor_colab_bundle_20260418" / "bundle_root"
if not bundle_root.exists():
    candidates = list(extract_root.glob("**/bundle_root"))
    if not candidates:
        raise RuntimeError("Could not locate extracted bundle_root folder.")
    bundle_root = candidates[0]

print("Zip path:", zip_path)
print("Extracted bundle_root:", bundle_root)
bundle_root


Upload required: oi_round2_post_advisor_colab_bundle_20260418.zip


KeyboardInterrupt: 

In [ ]:
import subprocess
import sys

required_packages = ["numpy", "pandas", "scipy", "statsmodels"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required_packages])
print("Installed/verified packages:", ", ".join(required_packages))


In [ ]:
import os
from pathlib import Path

os.chdir(bundle_root)
print("Working directory set to:", Path.cwd())

entry_script = Path("02_analysis/scripts/validation/oi_oro_dental_post_advisor_round2_reanalysis_v1.py")
dataset_path = Path("01_data/derived/osteogenesis_imperfecta_analysis_ready_post_advisor_round2_v1_2026-04-18.csv")

assert entry_script.exists(), f"Missing full script: {entry_script}"
assert dataset_path.exists(), f"Missing dataset: {dataset_path}"
print("Entry script:", entry_script)
print("Dataset:", dataset_path)


In [ ]:
import subprocess
import sys

run_cmd = [sys.executable, str(entry_script)]
print("Running full script:", " ".join(run_cmd))
subprocess.check_call(run_cmd)
print("Full script execution completed.")


In [ ]:
from pathlib import Path

output_dir = Path("03_outputs/reports/run_20260418_1037_post_advisor_round2")
expected_outputs = [
    output_dir / "primary_results_table.csv",
    output_dir / "robustness_loo_results.csv",
    output_dir / "run_manifest.json",
]

missing = [str(p) for p in expected_outputs if not p.exists()]
if missing:
    raise RuntimeError("Missing expected outputs: " + ", ".join(missing))

print("Output folder exists:", output_dir.exists())
for p in expected_outputs:
    print("OK", p)
